In [27]:
import zipfile
import numpy as np
import math
import pickle
import json
from scipy.sparse import load_npz

In [11]:
matrix = load_npz("matrices/sparse_matrix_train.npz")

# Queremos saber las canciones más populares, así que sumamos el valor total de cada columna

In [12]:
track_count = matrix.sum(axis=0) # sumamos por columna

track_count = np.asarray(track_count).flatten() # lo transformamos a 1D, que antes estaba como (1, 2262292)

### Ordenamos de más popular a menos, pero lo que nos interesa no es el valor, sino el índice para poder mapearlo, por eso usamos argsort

In [13]:
idx_sorted = np.argsort(track_count)[::-1] # el -500: es para hacer los últimos 500 y el ::-1 es para ordenarlo al revés

In [ ]:
# valores = np.sort(track_count)[-10:][::-1] # por si quisiéramos ver el valor, el número de veces que se guardó esa canción

# Cargamos el mapa de tracks para relacionar el índice con track_uri

In [14]:
with open("mapas/tracks_train.pkl", "rb") as f:
    track_to_idx = pickle.load(f)

In [15]:
idx_to_track = {v: k for k, v in track_to_idx.items()}

In [16]:
n = 1000
top_n_idx = idx_sorted[:n]
top_n_tracks = [idx_to_track[idx] for idx in top_n_idx]

# Leemos los datos de test

In [2]:
path_test = "datos/spotify_test_playlists.zip"

In [ ]:
try:
    with zipfile.ZipFile(path_test, "r") as zipf:
        for filename in zipf.namelist():
            if filename.endswith("test_input_playlists.json"):
                with zipf.open(filename, "r") as f:
                    test_data = json.load(f)
except Exception as e:
    print(e)

hola
hola


In [ ]:
pids = []
for playlist in test_data["playlists"]:
    pid = playlist["pid"]
    pids.append(pid)


In [22]:
a = []
n = set(a)
print(n)
print(type(n))
for i in n:
    print(i)

set()
<class 'set'>


In [ ]:
total_recommendations = []
for playlist in test_data["playlists"]:

    playlist_recommendations = [playlist["pid"]]
    playlist_tracks = []

    for t in playlist["tracks"]:
        playlist_tracks.append(t["track_uri"])

    playlist_tracks = set(playlist_tracks)
    
    while len(playlist_recommendations)<501: # esto está mal
        for top_track in top_n_tracks:
            if top_track not in playlist_tracks:
                playlist_recommendations.append(top_track)

    total_recommendations.append(playlist_recommendations)


In [30]:
print(len(total_recommendations[1]))

1001


In [28]:
def r_precision(predicted, actual):
    """Calcula R-Precision: % de aciertos ajustado al tamaño de la verdad."""
    if not actual: return 0.0
    # Solo miramos tantas predicciones como canciones reales faltan (R)
    r = len(actual)
    predicted_r = set(predicted[:r])
    actual_set = set(actual)
    
    intersection = predicted_r.intersection(actual_set)
    return len(intersection) / r

def dcg(relevance_scores, k):
    """Discounted Cumulative Gain"""
    relevance_scores = np.asarray(relevance_scores, dtype=float)[:k]
    
    if relevance_scores.size == 0:
        return 0.0
    
    # Formula: rel_i / log2(i + 2)
    return np.sum(relevance_scores / np.log2(np.arange(2, relevance_scores.size + 2)))

def ndcg(predicted, actual, k=500):
    """Normalized DCG: Qué tan bien ordenadas están las recomendaciones."""
    if not actual: return 0.0
    
    actual_set = set(actual)
    # Creamos un vector de 1s y 0s: 1 si la canción predicha es correcta, 0 si no
    relevance = [1 if x in actual_set else 0 for x in predicted]
    
    # DCG real (nuestra puntuación)
    dcg_val = dcg(relevance, k)
    
    # IDCG ideal (puntuación perfecta)
    ideal_relevance = [1] * len(actual) + [0] * (k - len(actual))
    idcg_val = dcg(ideal_relevance, k)
    
    if idcg_val == 0: return 0.0
    return dcg_val / idcg_val

def recommended_songs_clicks(predicted, actual):
    """
    Métrica específica de Spotify: 
    ¿Cuántos 'refrescos' de 10 canciones necesita el usuario 
    para encontrar la primera relevante?
    """
    if not actual: return 51 # Valor de penalización máximo + 1
    
    actual_set = set(actual)
    for i, track in enumerate(predicted):
        if track in actual_set:
            # Si está en la pos 0-9 -> 1 click
            # Si está en la pos 10-19 -> 2 clicks, etc.
            return math.floor(i / 10) + 1
            
    return 51 # Si no encontramos nada en las 500

In [29]:
predictions_path = "resultado_baseline.json" 
ground_truth_zip = "datos/spotify_test_playlists.zip" 

In [ ]:
# Cargar Predicciones
with open(predictions_path, 'r') as f:
    preds_data = json.load(f)

In [ ]:
# Cargar Ground Truth
with zipfile.ZipFile(ground_truth_zip, 'r') as zipf:
    for filename in zipf.namelist():
        if filename.endswith("test_eval_playlists.json"):
            with zipf.open(filename) as f:
                gt_data = json.loads(f.read())
            break

# Convertir GT a formato diccionario fácil si no lo es
gt_dict = {}
if "playlists" in gt_data:
    for pl in gt_data["playlists"]:
        # Extraemos SOLO las URIs de los tracks ocultos
        gt_dict[pl["pid"]] = [t["track_uri"] for t in pl["tracks"]]
else:
    print("Formato de test_eval.json desconocido.")
    exit()

print(f"Evaluando {len(preds_data)} playlists...")

r_precisions = []
ndcgs = []
clicks = []

for pid_str, predicted_tracks in preds_data.items():
    # El PID en el json a veces es string, a veces int. Aseguramos consistencia.
    pid = int(pid_str) 
    # Buscamos este PID en la verdad
    actual_tracks = gt_dict.get(pid) or gt_dict.get(str(pid))
    
    if actual_tracks is None:
        continue # Si no hay datos de verdad para esta playlist, saltamos
    
    # Calcular métricas para esta playlist
    rp = r_precision(predicted_tracks, actual_tracks)
    nd = ndcg(predicted_tracks, actual_tracks)
    cl = recommended_songs_clicks(predicted_tracks, actual_tracks)
    
    r_precisions.append(rp)
    ndcgs.append(nd)
    clicks.append(cl)

# RESULTADOS FINALES
print(" EVALUACIÓN ITERACIÓN 0 \n")
print(f"R-Precision media: {np.mean(r_precisions):.4f}")
print(f"NDCG media:        {np.mean(ndcgs):.4f}")
print(f"Clicks medios:     {np.mean(clicks):.4f}")